## Bath_dMFA_2026 Software 2

## Passenger vehicles in the EUR20 countries - From inflow to stock

**This software exercise covers calculations of the stock of vehicles in the use phase (accumulation of inflow) and the outflow of end-of-life vehicles from the stock.** It applies the inflow-driven dynamic stock model and serves as preparation of the stock-driven model in the subsequent workbooks.

The data workbook *Bath_dMFA_2026_Software_Data.xlsx* contains a number of parameters for the calculation: Historic inflow of new registration of passenger vehicles in the EUR20 countries from 1990 to 2024, the split of total new registration into gasoline and electric vehicles, and the average lifetime of vehicles in the use phase.

**Using the material from the previous software workbook (data and code), work on completing the following tasks to calculate and interpret the following indicators/quantities**

### Task 1: For the historic data, what are the time series for the inflow of new vehicles into the stock, broken down by drive technology?
Calculate the relevant flows using the Python code below.
Split the inflow into electric and gasoline vehicles as before.

In [ ]:
# First, import required libraries:
import pandas as pd   # dataframe library, used for fast import/export from/to excel and for organizing the data.
import numpy as np    # math library
import matplotlib.pyplot as plt  # plotting library
import scipy # library with statistical functions

Read input data from excel into dataframe and convert to numpy array.
Determine and print the shape of the different arrays (_.shape_ function).

In [ ]:
# First, the historic inflow:
I_c = pd.read_excel('Bath_dMFA_2026_Software_Data.xlsx', sheet_name='New registration', index_col=0).values[:,0]
print(I_c.shape) # rows: 35 years 1990-2024

In [ ]:
# Second, the historic split of the inflow into different drive technologies:
BEVshare_cT = pd.read_excel('Bath_dMFA_2026_Software_Data.xlsx', sheet_name='BEV_share', index_col=0).values
print(BEVshare_cT.shape) # rows: 71 years 1990-2060, columns: 2 drive technologies

In [ ]:
# Third, the lifetime:
Lifetime_T = pd.read_excel('Bath_dMFA_2026_Software_Data.xlsx', sheet_name='Lifetime', index_col=0).values.transpose()
# For the subsequent calculation of the probability density function, the lifetime list must be a row vector, that's why the .transpose() command.
print(Lifetime_T.shape) # rows: aggregate for all age-cohorts, columns: 2 drive technologies

We first split the inflow into the two drive technologies as before (copy from the previous workbook):

In [ ]:
I_cT = np.einsum('c,cT->cT',I_c,BEVshare_cT[0:35,:]) / 100 

With this preparation, we are ready to tackle the core tasts of this workbook:

### Task 2: Build and analyse the probability density functions (pdf) of discarding the vehicles after use, for each drive technology 

The following equation shows the normally distributed probability of discard "normally distributed lifetime), with mean lifetime $\tau$ and with a standard deviation $\sigma$, a commonly used lifetime model.

$$ pdf(age) = \frac{1}{\sigma \sqrt{2\pi} } e^{-\frac{1}{2}\left(\frac{age-\tau}{\sigma}\right)^2} $$

In the code below, we first generate the pdf function from the lifetime parameter, assuming a normally distributed lifetime with a standard deviation of 30% of the mean, a commonly used assumption.

In [ ]:
print(Lifetime_T)

pdf = scipy.stats.norm.pdf(np.arange(0,35,1),Lifetime_T,0.3*Lifetime_T) # pdf of the normal distribution with mean lifetime and standard deviation 0.3*lifetime

print(pdf.shape)
print(pdf)

plt.plot(pdf.transpose()) # transpose to have time dimension on x axis, no fan.cy labels and legend here, just a quick look

The curve for gasoline vehicles (blue) is further to the right (indicating a longer lifetime on average) and a slightly wider spread. The curve for BEVs is a bit more narrow and further to the left, indicating a shorter average lifetime.
The area under each curve is 1, indicating that the total propability of the inflow leaving the stock eventually is 100%.

In [ ]:
# Check that the pdf sum up to 1 by using the np.cumsum function:
plt.plot(np.cumsum(pdf.transpose(),axis=0)) 
# transpose to have the years on the x axis, then determine the cumulative sum alon this axis.

#### Task 2.1: Repeat the calculation of the lifetime for a wider distribution, e.g., standard deviation = 0.7 * average lifetime!

In [ ]:
pdf_wide = ...

#### Task 2.2: Repeat the calculation of the lifetime for a narrower distribution, e.g., standard deviation = 0.1 * average lifetime!

In [ ]:
pdf_narrow = ...

Note that the wider the lifetime distribution curve, the lower the values for the individual years, as the probability of discard is smoothed out across more years than with a more narrow distribution, and vice versa.

### Task 3: For the historic data: Estimate the outflow of old vehicles from the stock, and the stock (fleet size) itself, broken down by age-cohorts! Inspect and interpret the results!

We apply the **inflow-driven dynamic stock model** to tackle this question.

First, we define the result arrays for stock _S_ and outflow _O_ in their highest resolution, as matrices with the three dimensions time _t_, age-cohort _c_, and technology _T_.

We need the breakdown of all variables into age-cohorts because technology parameters, such as specific energy consumption and material composition, depend on the age-cohort of the vehicle.

In [ ]:
# Define result arrays with maximum detail, tracing the different age-cohort separately:
S_tcT  = np.zeros((35,35,2)) # 3D stock table        for time t, age-cohort c, and drive technology T.
DS_tcT = np.zeros((35,35,2)) # 3D stock change table for time t, age-cohort c, and drive technology T.
O_tcT  = np.zeros((35,35,2)) # 3D outflow matrix     for time t, age-cohort c, and drive technology T.

Then, an implementation of the inflow-driven model is presented and commented below.
This implementation follows the three steps given in the lecture slides:
1. compute outflow from pdf * inflow
2. compute stock change from balancing equation
3. compute stock as cumulative summ of stock change

In [ ]:
# Define result arrays with maximum detail, tracing the different age-cohort separately:
S_tcT  = np.zeros((35,35,2)) # 3D stock table        for time t, age-cohort c, and drive technology T.
DS_tcT = np.zeros((35,35,2)) # 3D stock change table for time t, age-cohort c, and drive technology T.
O_tcT  = np.zeros((35,35,2)) # 3D outflow matrix     for time t, age-cohort c, and drive technology T.

# The pdf indicates the probability of outflow of a given age-cohort.
# We can compute the stock change table and the outflow with a double for loop, the first one for the historic years 
# and the inner one for the different years until each year, the different age-cohorts:

# Define ancillary data container: The vehicles from different age-cohorts leaving in a given year:
veleave_t_c = np.zeros((35)) # length: number of age-cohorts

# Populate the outflow table from inflow
# Populate the stock change table from the balancing equation
for drivetech in range(0,2): # for all drive technologies
    for year in range(0,35): # for all historic years
        veleave_t_c = np.zeros((35)) # empty cohort-specific outflow container for this year
        for age_cohort in range(0,year+1): # for all inflow years till year (need to add 1 since Python stops counting 1 place before the right boundary)
            # Calculate leaving vehicles in given year, for the different age-cohorts:
            veleave_t_c[age_cohort] = I_cT[age_cohort,drivetech] * pdf[drivetech,year-age_cohort] 
        # Add the leaving vehicles to outflow matrix:
        O_tcT[year,:,drivetech]    += veleave_t_c # add leaving vehicles in place  
        # Compute the stock change of the given year, using the balancing equation
        DS_tcT[year,year,drivetech] = I_cT[year,drivetech]    # first, add the inflow for the given year = age-cohort 
        DS_tcT[year,:,drivetech]   -= O_tcT[year,:,drivetech] # substract outflow in place

# Calculate the stock as cumulative sum of the stock change
S_tcT = np.cumsum(DS_tcT, axis = 0)

# This completes the inflow-driven model.

Next, we inspect the main result arrays for stock and outflow, covering both gasoline and electric vehicles.

In [ ]:
plt.imshow(S_tcT[:,:,0],interpolation='nearest') # create a simple heatmap: larger value --> brighter color.
plt.xlabel('age-cohort')    # add x axis label 
plt.ylabel('year')          # add y axis label
plt.title('Stock of gasoline cars, by year and by age-cohort') # add title
plt.show(); # show plot with all features from above

#### Task 3.1: Create a corresponding visualisation for the BEV stock!

In [ ]:
plt.imshow(...

Let's look at the outflow now:

In [ ]:
plt.imshow(O_tcT[:,:,0],interpolation='nearest') # create a simple heatmap: larger value --> brighter color.
plt.xlabel('age-cohort')    # add x axis label 
plt.ylabel('year')          # add y axis label
plt.title('Outflow of gasoline vehicles, by year and by age-cohort') # add title
plt.show(); # show plot with all features from above

#### Task 3.2: Create a corresponding visualisation for the BEV outflow!

In [ ]:
plt.imshow(...

With the information above, the following questions can now be answered:

**What is the meaning of the data shown above?** The stock table shows the size of the fleet of passenger vehicles for the EUR20 countries for each historic year from 1990 to 2024, given by the row index. For each year, the fleet consists of vehicles from different age-cohorts (year of manufacturing). For each year (row), the fleet can therefore be broken down into different age-cohorts (column index), showing the manufacturing year (from 1990 to 2024). The new cars of each year are on the diagonal, where year = age-cohort. 

**What is the meaning of the row index, and what is the meaning of the column index?** 
The row indicates the historic year, and the column the year of manufacturing (age-cohort). 

**Exactly how large was the car fleet in 2022?** to answer this question, we take the corresponding row of the table and sum it up. In Python, this is done as follows:

In [ ]:
# First, select the row for 2022, which is the 32nd row, index 32, since in Python, we start counting at 0 (for 1990).
# Then, selet all columns (all age-cohorts) and drive technologies, indicated by ':'.
# Lastly, sum up over this table row by the .sum() command:
Stock_2022 = S_tcT[32,:,:].sum()
Stock_2022 # print the total stock in 2022, unit: vehicles

The total fleet size in the EUR20 countries in 2022 was about 161 million passenger vehicles.

Below, we plot the time series of the **total fleet size**:

In [ ]:
plt.plot(np.arange(1990,2025,1), np.einsum('tcT->t',S_tcT))
plt.title('Evolution of the vehicle fleet (in-use stock), EUR20 countries')
plt.ylabel('Number of vehicles, unit: 1')
plt.xlabel('Year')

**How many cars in 2022 were 10 years old or younger?**

In [ ]:
# In 2022, the oldest age-cohort (1990) was 32 years old, so we need to slice all age-cohorts from 2012 (index 22) onwards:
Stock_2022_That_is_younger_than_10_years = S_tcT[32,22::,:].sum()
Stock_2022_That_is_younger_than_10_years

In 2022, there were about 102 million vehicles in the EUR20 countries that were 10 years old or younger.

**What is the meaning of a given column (age-cohort) in this table?**
A given table row traces a given age-cohort over time. The following plot illustrates this behaviour of the age-cohort of 1995:

In [ ]:
np.arange(1990,2025,1)

In [ ]:
S_tcT[:,5,0]

#### Task 3.3: Create a visualisation for the 1995 age-cohort in the EUR20 passenger vehicle fleet!

In [ ]:
plt.plot(...

#### Task 3.4: Create a visualisation of the end-of-life vehicles (outflow) since 1990 and starting with the 1990 age-cohort from the EUR20 passenger vehicle fleet! How many cars from the inflow since 1990 have already left the fleet?

To calculate the total outflow (all age-cohors and drive technologies in a given year), calculate a suitable aggregation of _O_tcT_ using the np.einsum function.

In [ ]:
# We'll address the second question first, simply by adding up the total outflow:
Total_Outflow = ...
Total_Outflow

Out of the 365 million vehicles that were registered since 1990, 208 million have already left the stock again, that is 208 million passenger vehicles over the last 35 years that were either exported to other world regions or send to the waste management industries for shredding.

We contine with plotting the outflow:

In [ ]:
plt.plot(...

The outflow has risen steadily and recently stabilized at about 11 million vehicles/yr, stabilizin the fleet size at 155...175 million units. 

Note that pre-1990 age-cohorts are not covered here, which is why this outflow only covers part of the total outflow (the post 1989 age-cohorts).

Since about 2020, the pre 1990 age-cohorts don't play a role anymore, and the 1900 truncation is not relevant anymore, the stock by age-cohort is accurate enough to use it for further modelling.

### Task 4: Show that inflow, stock, and outflow are balanced for vehicles in each year!

In [ ]:
# First, sum up over all age-cohorts, using the einsum function:
S_tT  = np.einsum('tcT->tT',S_tcT)
O_tT  = np.einsum('tcT->tT',O_tcT)
DS_tT = np.diff(S_tT,axis = 0, prepend=0) # re-calculate the stock change from the stock, starting with 0 in the first year (prepdent = 0)

Balance = ...

In [ ]:
np.abs(Balance).sum() # add up all absolute values in this balance, should be 0 or at least close to it.

The overall added imbalance is less than a millionth of a single car, from a practical perspective, the dynamic stock model is hence balanced.

### Task 5: Store the stock and outflow tables for further use!

We'll need the 1900-2025 stock table to initialize the stock-driven model in the subsequent workbooks.

In [ ]:
pd_xlsx_writer = pd.ExcelWriter('Bath_dMFA_2026_Software_2_Results.xlsx')

In [ ]:
export_df1= pd.DataFrame(data=S_tcT[:,:,0],   # entire t x c stock table for gasoline cars
            index  =np.arange(1990,2025,1),   # 1st column as index
            columns=np.arange(1990,2025,1))   # 1st row as the column names

export_df1.to_excel(pd_xlsx_writer, sheet_name="Stock_tc_gasoline", merge_cells=False)   

export_df2= ...

In [ ]:
pd_xlsx_writer.close()

This export concludes this exercise.